<a href="https://colab.research.google.com/github/simon-clematide/colab-notebooks-for-teaching/blob/main/spam_or_ham_sms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📩 Welcome to the SMS Spam Classifier Tutorial!

Have you ever wondered how your phone automatically moves annoying promotional texts into a "Junk" or "Spam" folder? Today, you are going to build exactly that!

In this beginner-friendly tutorial, we will use the famous **SMS Spam Collection Dataset** to teach an AI how to distinguish between normal text messages ("Ham") and unwanted commercial messages ("Spam").

### What we will do:
1. **Load and Explore:** Look at real SMS text messages.
2. **Feature Extraction:** Convert English words into mathematical patterns a computer can understand (using a basic Natural Language Processing technique called Bag-of-Words).
3. **Train the Brain:** Teach a **Naive Bayes** machine learning model to spot the difference between Spam and Ham.
4. **Test the AI:** Give the model brand-new messages to see how smart it really is!

No prior coding or math experience is needed to understand the concepts. Let's get started! 🚀

### Step 1: Gather the Tools and Raw Data
Before we can build our spam filter, we need to gather our tools and the raw material (the dataset). We'll download the SMS Spam Collection, which contains thousands of real text messages.

In [ ]:
# @title
import pandas as pd
import urllib.request
import zipfile
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score

# Download and extract the dataset from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
zip_path = "smsspamcollection.zip"
if not os.path.exists("SMSSpamCollection"):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
print("Setup complete!")

### Step 2: Look at the Data
Now that we have the file, we need to open it and look inside. We load the messages into a virtual table so our program can easily read and manipulate them.

In [ ]:
# Load the data (tab-separated, no headers)
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])
print("Dataset loaded successfully! Here are the first 5 rows:")
display(df.head(n=10))

### Step 2.5: Subsample to Balance the Dataset
Our dataset has a strong class imbalance: there are 4,827 legitimate messages ("ham") and only 747 spam messages. If we kept it this way, a lazy model that predicts "ham" 100% of the time would still get ~86.6% accuracy!

To prevent this, we'll subsample our dataset so we have an equal number of ham and spam messages (747 of each).

In [ ]:
# Separate spam and ham
spam_df = df[df['label'] == 'spam']
ham_df = df[df['label'] == 'ham']

# Subsample ham to match the count of spam
ham_subsampled = ham_df.sample(n=len(spam_df), random_state=42)

# Combine back into a balanced dataset
df_balanced = pd.concat([spam_df, ham_subsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

# Assign back to df for the subsequent steps
df = df_balanced

print(f"New balanced dataset shape: {df.shape}")
print(df['label'].value_counts())

### Step 3: Prepare the Data (Train, Dev, and Test)
Computers understand math much better than English text. First, we convert our "ham" and "spam" labels into `0`s and `1`s.

Then, like good scientists, we split our data into three distinct parts:
1. **Training Set (80%)**: The textbook the AI uses to learn.
2. **Dev Set (10%)**: A "practice quiz." We use this to experiment with settings (like lowercasing) without cheating on the final exam.
3. **Test Set (10%)**: The "final exam." We lock this away and only use it at the very end to check true performance.

In [ ]:
# @title
# Convert text labels into numbers: ham = 0, spam = 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['message']
y = df['label_num']

# First, split into Training (80%) and a Temporary Set (20%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Next, split the Temporary Set exactly in half to get Dev (10%) and Test (10%)
X_dev, X_test, y_dev, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Data split complete!\nTrain size: {len(X_train)}\nDev size: {len(X_dev)}\nTest size: {len(X_test)}")

### Step 4: The Experiment (Using the Dev Set)
Should we convert all text to lowercase? Let's run a scientific experiment!

We will create two versions of our AI: one that lowercases words, and one that leaves them exactly as typed. We will test both on our **Dev Set** practice quiz to see which approach scores higher.

In [ ]:
# @title
def test_lowercasing(lowercase_setting):
    # 1. Feature Extraction (Bag-of-Words)
    # Match words (\w+) OR sequences of punctuation/emojis ([^\w\s]+)
    vectorizer = CountVectorizer(
        lowercase=lowercase_setting,
        max_features=2000,
        token_pattern=r"(?u)\w+|[^\w\s]+"
    )
    X_train_vec = vectorizer.fit_transform(X_train)
    X_dev_vec = vectorizer.transform(X_dev)

    # 2. Train the Model
    model = BernoulliNB()
    model.fit(X_train_vec, y_train)

    # 3. Grade on Dev Set
    y_dev_pred = model.predict(X_dev_vec)
    acc = accuracy_score(y_dev, y_dev_pred) * 100
    print(f"Accuracy with lowercase={lowercase_setting}: {acc:.3f}%")

print("--- Running Dev Set Experiment ---")
test_lowercasing(lowercase_setting=True)
test_lowercasing(lowercase_setting=False)

### Step 5: Train the Final AI
Now we look at our experiment results. Usually, they are very close, but `lowercase=True` is the standard best practice in NLP because it simplifies the patterns the AI has to learn.

Let's build our official, final model using `lowercase=True`.

In [ ]:
# @title
# Train the final model with our chosen settings
final_vectorizer = CountVectorizer(
    lowercase=False,
    max_features=2000,
    token_pattern=r"(?u)\w+|[^\w\s]+"
)
X_train_vec_final = final_vectorizer.fit_transform(X_train)

final_model = BernoulliNB()
final_model.fit(X_train_vec_final, y_train)

print("Final model trained successfully!")

### Step 6: The Final Exam (Test Set)
The time has come! We grade our final AI on the **Test Set** (the 10% of messages it has never seen, not even during the experiment). This gives us our most honest, true accuracy score.

In [ ]:
# Evaluate the Final Model on the Test Set
X_test_vec = final_vectorizer.transform(X_test)
y_test_pred = final_model.predict(X_test_vec)

print(f"Final Accuracy on Test Data: {accuracy_score(y_test, y_test_pred) * 100:.2f}%")

### Step 7: Try it Yourself!
The model is ready! Let's give it some brand new, made-up text messages and see if it can correctly figure out which ones are legitimate and which ones are spam.

In [ ]:
# Test it yourself with new messages!
sample_messages = [
    "Hey, are we still on for dinner tonight?",
    "CONGRATULATIONS! You have won a $1,000 Walmart gift card. Click here to claim your prize.",
    "Don't forget to bring the documents to the meeting.",
    "URGENT: Your account has been suspended. Reply with your PIN to reactivate."
]

# Convert new text to numbers using the final_vectorizer
sample_vec = final_vectorizer.transform(sample_messages)
predictions = final_model.predict(sample_vec)

print("--- Testing with New Messages ---")
for msg, pred in zip(sample_messages, predictions):
    label = "SPAM" if pred == 1 else "HAM (Legitimate)"
    print(f"Message: '{msg}'\nPrediction: {label}\n")

### Step 8: Peek Inside the AI's Brain (Feature Explanation)
How exactly is the AI making its decisions? Machine learning isn't just magic; it's just math!

Let's extract the "feature weights" (the mathematical scores) the AI assigned to each word and punctuation mark during training. This will show us exactly which tokens the model considers the most "Spammy" and which ones it considers normal ("Hammy").

In [ ]:
# @title
import numpy as np

# 1. Get the list of all 2,000 words/tokens the AI learned
words = final_vectorizer.get_feature_names_out()

# 2. Get the model's internal scores for each word
# feature_log_prob_[1] is the score for Spam, [0] is for Ham
# We subtract Ham from Spam to find words that are overwhelmingly Spam-heavy
spam_scores = final_model.feature_log_prob_[1] - final_model.feature_log_prob_[0]

# 3. Put it in a nice table
feature_df = pd.DataFrame({
    'token': words,
    'spam_score': spam_scores
})

In [ ]:
print("🔥 Top 15 Most 'Spammy' Tokens:")
# Sort by highest spam score
display(feature_df.sort_values(by='spam_score', ascending=False).head(15))

print("\n🛡️ Top 15 Most 'Hammy' (Normal) Tokens:")
# Sort by lowest spam score (which means highest ham score)
display(feature_df.sort_values(by='spam_score', ascending=True).head(15))